# Load Colors from Rebrickable API
Fetches all color records from the Rebrickable REST API (paginated) and loads them into a Spark DataFrame.

## Imports and Configuration

Import standard and third-party libraries needed for HTTP requests, JSON handling, Spark processing, and Delta Lake operations. `BASE_URL` points to the Rebrickable colors endpoint; `REQUEST_DELAY_SECONDS` keeps requests within the ~1 req/sec rate limit.

In [ ]:
import json
import time
import requests
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

API_KEY               = ""
BASE_URL              = "https://rebrickable.com/api/v3/lego/colors/"
PAGE_SIZE             = 1000
REQUEST_DELAY_SECONDS = 1.1  # stay within ~1 req/sec rate limit

## Fetch Colors from API

Paginate through the Rebrickable colors endpoint, collecting all records into a list. The row count is printed after each page so you can monitor progress, and a short delay between requests respects the API rate limit.

In [ ]:
def fetch_all_colors(api_key: str, base_url: str, page_size: int, delay: float) -> list:
    headers = {"Authorization": f"key {api_key}"}
    params = {"page": 1, "page_size": page_size}
    all_results = []

    next_url = base_url

    while next_url:
        response = requests.get(next_url, headers=headers, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()
        all_results.extend(data.get("results", []))

        next_url = data.get("next")  # None when there are no more pages
        params = {}  # next_url already contains all query params

        print(f"Fetched {len(all_results)} / {data['count']} records")

        if next_url:
            time.sleep(delay)

    return all_results


colors_data = fetch_all_colors(API_KEY, BASE_URL, PAGE_SIZE, REQUEST_DELAY_SECONDS)
print(f"Total colors fetched: {len(colors_data)}")

## Convert to Spark DataFrame

Define an explicit schema (`id`, `name`, `rgb`, `is_trans`, `external_ids`) and convert the API results to a typed Spark DataFrame. `external_ids` is serialised as a JSON string to keep the schema flat. Printing the schema and previewing 10 rows confirms the conversion looks correct before writing.

In [ ]:
schema = StructType([
    StructField("id",           IntegerType(), nullable=False),
    StructField("name",         StringType(),  nullable=True),
    StructField("rgb",          StringType(),  nullable=True),
    StructField("is_trans",     BooleanType(), nullable=True),
    StructField("external_ids", StringType(),  nullable=True),  # stored as JSON string
])

rows = [
    (
        record["id"],
        record.get("name"),
        record.get("rgb"),
        record.get("is_trans"),
        json.dumps(record.get("external_ids")) if record.get("external_ids") else None,
    )
    for record in colors_data
]

spark = SparkSession.builder.getOrCreate()
df_colors = spark.createDataFrame(rows, schema=schema)

df_colors.printSchema()
df_colors.display(10, truncate=False)

## Write Raw Parquet to External Volume

Append audit columns (`_load_datetime`, `_record_source`) and write the data as a single Parquet file to a date/time-partitioned path under the Unity Catalog external volume. The `part-` file is renamed to a deterministic filename and the temporary staging directory is removed.

In [ ]:
# Generate timestamp components for partitioned path and filename
now   = datetime.utcnow()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")
ts    = now.strftime("%Y%m%d_%H%M%S")

# Add audit columns
df_colors_out = (
    df_colors
    .withColumn("_load_datetime", current_timestamp())
    .withColumn("_record_source", lit("API"))
)

# Update the base path below to match your Unity Catalog external volume mount point
EXTERNAL_VOLUME_BASE = "/Volumes/lego_brickbase/bronze/raw_data_volume/colors"
PARTITION_PATH       = f"{EXTERNAL_VOLUME_BASE}/{year}/{month}/{day}/{ts}"
TEMP_PATH            = f"{PARTITION_PATH}/_tmp"
FILE_NAME            = f"raw_colors_{ts}.parquet"
FINAL_PATH           = f"{PARTITION_PATH}/{FILE_NAME}"

# Write as a single Parquet file to a temp staging directory
df_colors_out.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(TEMP_PATH)

# Locate the single part file and rename it to the desired filename
part_files = [f.path for f in dbutils.fs.ls(TEMP_PATH) if f.name.startswith("part-")]
dbutils.fs.mv(part_files[0], FINAL_PATH)
dbutils.fs.rm(TEMP_PATH, recurse=True)

print(f"Colors written to: {FINAL_PATH}")

## Merge into SCD2 Delta Table

Implement a Type 2 Slowly Changing Dimension (SCD2) merge against the Delta table:

1. **First load** — all incoming rows are written as current records.
2. **Subsequent loads** — a two-step merge is applied:
   - *Step 1:* Expire changed records (`valid_to` = now, `is_current` = false) and soft-delete rows no longer present in the source (`is_deleted` = true).
   - *Step 2:* Insert new records and new versions of changed records as current rows.

Finally, the Delta table is registered in Unity Catalog under `lego_brickbase.bronze.raw_colors` if it does not already exist.

In [ ]:
DELTA_TABLE_PATH = "/Volumes/lego_brickbase/bronze/delta_volume/colors"
CATALOG_TABLE    = "lego_brickbase.bronze.raw_colors"

# Natural key and tracked business columns
KEY_COLS     = ["id"]
TRACKED_COLS = ["name", "rgb", "is_trans", "external_ids"]

# Read the Parquet file written in the previous step
df_source = spark.read.parquet(FINAL_PATH)

# Attach SCD2 metadata columns
df_incoming = (
    df_source
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to",   lit(None).cast(TimestampType()))
    .withColumn("is_current", lit(True))
    .withColumn("is_deleted", lit(False))
)

if not DeltaTable.isDeltaTable(spark, DELTA_TABLE_PATH):
    # First load — write all records as current
    df_incoming.write \
        .format("delta") \
        .mode("overwrite") \
        .save(DELTA_TABLE_PATH)
    print(f"Delta table created at {DELTA_TABLE_PATH}")
else:
    delta_table    = DeltaTable.forPath(spark, DELTA_TABLE_PATH)
    join_condition = " AND ".join([f"tgt.{c} = src.{c}" for c in KEY_COLS])
    change_cond    = " OR ".join([f"tgt.{c} != src.{c}" for c in TRACKED_COLS])

    # Step 1: Expire changed records; soft-delete records missing from source
    (
        delta_table.alias("tgt")
        .merge(df_incoming.alias("src"), f"{join_condition} AND tgt.is_current = true")
        .whenMatchedUpdate(
            condition=change_cond,
            set={"valid_to": "src.valid_from", "is_current": "false"}
        )
        .whenNotMatchedBySourceUpdate(
            condition="tgt.is_current = true",
            set={"valid_to": "current_timestamp()", "is_current": "false", "is_deleted": "true"}
        )
        .execute()
    )

    # Step 2: Insert new records and new versions of changed records
    df_current   = delta_table.toDF().filter("is_current = true")
    df_to_insert = df_incoming.join(df_current.select(*KEY_COLS), on=KEY_COLS, how="left_anti")
    df_to_insert.write.format("delta").mode("append").save(DELTA_TABLE_PATH)

row_count = DeltaTable.forPath(spark, DELTA_TABLE_PATH).toDF().filter("is_current = true").count()
print(f"Active rows in Delta table: {row_count:,}")

# Register the Delta table in Unity Catalog
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
    AS SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")
print(f"Catalog table ready: {CATALOG_TABLE}")